<a href="https://colab.research.google.com/github/Emad-Biotech540/PD1-Immunotherapy-Bioinformatics-Analysis./blob/main/RNAseq_Differential_Expression_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# في بيئة R على Colab
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

    # تثبيت الحزم الأساسية للتحليل والرسم
    BiocManager::install(c("DESeq2", "EnhancedVolcano", "clusterProfiler", "pheatmap"))
    install.packages(c("ggplot2", "ggrepel", "pheatmap"))

In [ ]:
# التحقق من نجاح تحميل الحزم
library(DESeq2)
library(EnhancedVolcano)
library(pheatmap)
library(clusterProfiler)

print("✅ تم تثبيت جميع الحزم بنجاح! أنا جاهز للتحليل.")

In [ ]:
# ============================================
# المرحلة 1: إنشاء بيانات RNA-seq تجريبية
# ============================================

set.seed(42)

# 1. تحديد معاملات البيانات
n_genes <- 2000      # عدد الجينات
n_samples <- 12      # عدد العينات
n_control <- 6       # عدد عينات الكنترول
n_treatment <- 6     # عدد عينات العلاج

# 2. أسماء الجينات والعينات
gene_names <- paste0("Gene_", sprintf("%04d", 1:n_genes))
sample_names <- c(
  paste0("Control_", 1:n_control),
    paste0("Treatment_", 1:n_treatment)
    )

    # 3. إنشاء بيانات العد (Counts) - توزيع سالب ثنائي الحد
    # معظم الجينات تعبيرها منخفض، وبعضها مرتفع
    base_expression <- rnbinom(n_genes, mu = 100, size = 2)

    # 4. إنشاء مصفوفة العد
    counts_matrix <- matrix(0, nrow = n_genes, ncol = n_samples)
    rownames(counts_matrix) <- gene_names
    colnames(counts_matrix) <- sample_names

    for (i in 1:n_genes) {
      counts_matrix[i, ] <- rnbinom(n_samples, mu = base_expression[i], size = 3)
      }

      # 5. إضافة تأثير العلاج (Differential Expression)
      # نجعل 10% من الجينات upregulated و 10% downregulated
      n_de <- round(n_genes * 0.2)
      up_genes <- sample(1:n_genes, n_de/2)
      down_genes <- sample(setdiff(1:n_genes, up_genes), n_de/2)

      # مضاعفة التعبير في مجموعة العلاج
      counts_matrix[up_genes, 7:12] <- counts_matrix[up_genes, 7:12] * 3
      counts_matrix[down_genes, 7:12] <- round(counts_matrix[down_genes, 7:12] / 3)

      # 6. إنشاء metadata (معلومات العينات)
      sample_info <- data.frame(
        sample = sample_names,
          condition = factor(c(rep("Control", n_control), rep("Treatment", n_treatment)),
                               levels = c("Control", "Treatment")),
                                 batch = factor(rep(c("Batch1", "Batch2"), 6))
                                 )
                                 rownames(sample_info) <- sample_names

                                 # 7. عرض ملخص البيانات
                                 cat("✅ تم إنشاء البيانات التجريبية!\n")
                                 cat("========================================\n")
                                 cat("عدد الجينات:", n_genes, "\n")
                                 cat("عدد العينات:", n_samples, "\n")
                                 cat("عينات الكنترول:", n_control, "\n")
                                 cat("عينات العلاج:", n_treatment, "\n")
                                 cat("جينات Upregulated:", length(up_genes), "\n")
                                 cat("جينات Downregulated:", length(down_genes), "\n")
                                 cat("========================================\n\n")

                                 # عرض أول 5 صفوف وأعمدة
                                 cat("📊 أول 5 صفوف من البيانات:\n")
                                 print(head(counts_matrix[, 1:6]))

                                 cat("\n📋 معلومات العينات:\n")
                                 print(sample_info)

                                 # حفظ البيانات في ملفات
                                 write.csv(counts_matrix, "counts_matrix.csv")
                                 write.csv(sample_info, "sample_info.csv")
                                 cat("\n💾 تم حفظ البيانات في: counts_matrix.csv و sample_info.csv\n")

                                 # حفظ قوائم الجينات الحقيقية للـ DE للتحقق لاحقاً
                                 true_de_genes <- list(up = up_genes, down = down_genes)
                                 saveRDS(true_de_genes, "true_de_genes.rds")

In [ ]:
# ============================================
# المرحلة 2: إنشاء كائن DESeq2 وتحليل البيانات
# ============================================

library(DESeq2)

# 1. إنشاء كائن DESeq2
dds <- DESeqDataSetFromMatrix(
  countData = counts_matrix,
    colData = sample_info,
      design = ~ condition
      )

      # 2. تصفية الجينات منخفضة التعبير
      # نحتفظ فقط بالجينات التي لديها 10 قراءات على الأقل في 3 عينات
      keep <- rowSums(counts(dds) >= 10) >= 3
      dds <- dds[keep, ]

      cat("✅ عدد الجينات بعد التصفية:", nrow(dds), "\n")

      # 3. تشغيل التحليل التفاضلي
      cat("\n⏳ جاري تشغيل DESeq2...\n")
      dds <- DESeq(dds)

      # 4. استخراج النتائج
      res <- results(dds, contrast = c("condition", "Treatment", "Control"))
      res <- res[order(res$padj), ]

      cat("\n✅ اكتمل التحليل!\n")
      cat("========================================\n")
      cat("عدد الجينات المعنوية (padj < 0.05):", sum(res$padj < 0.05, na.rm = TRUE), "\n")
      cat("عدد الجينات Upregulated:", sum(res$padj < 0.05 & res$log2FoldChange > 1, na.rm = TRUE), "\n")
      cat("عدد الجينات Downregulated:", sum(res$padj < 0.05 & res$log2FoldChange < -1, na.rm = TRUE), "\n")
      cat("========================================\n\n")

      # عرض أهم 10 جينات
      cat("🏆 أهم 10 جينات معنوية:\n")
      print(head(res[!is.na(res$padj), c("baseMean", "log2FoldChange", "pvalue", "padj")], 10))

      # 5. تحويل البيانات للتطبيع (rlog)
      cat("\n⏳ جاري تحويل البيانات (rlog)...\n")
      rld <- rlog(dds, blind = FALSE)
      cat("✅ تم التحويل!\n")

      # حفظ النتائج
      saveRDS(dds, "dds.rds")
      saveRDS(res, "res.rds")
      saveRDS(rld, "rld.rds")
      cat("\n💾 تم حفظ النتائج!\n")

In [ ]:
# ============================================
# المرحلة 3: رسم جميع الرسومات البيانية
# ============================================

library(ggplot2)
library(pheatmap)
library(EnhancedVolcano)
library(RColorBrewer)

# تحميل الكائنات
dds <- readRDS("dds.rds")
res <- readRDS("res.rds")
rld <- readRDS("rld.rds")
sample_info <- read.csv("sample_info.csv", row.names = 1)

# ============================================
# 1. QC PLOTS (جودة البيانات)
# ============================================

cat("📊 رسم 1: Boxplot لتوزيع التعبير...\n")

# 1.1 Boxplot
png("01_Boxplot_QC.png", width = 1200, height = 800, res = 120)
par(mar = c(8, 4, 4, 2))
boxplot(log2(counts(dds) + 1),
        col = c(rep("#4A90E2", 6), rep("#E24A4A", 6)),
                las = 2, cex.axis = 0.7,
                        main = "Boxplot - Log2(Counts+1) per Sample",
                                xlab = "", ylab = "log2(Counts + 1)")
                                legend("topright", legend = c("Control", "Treatment"),
                                       fill = c("#4A90E2", "#E24A4A"), cex = 0.8)
                                       dev.off()

                                       cat("📊 رسم 2: Density Plot...\n")

                                       # 1.2 Density Plot
                                       png("02_Density_QC.png", width = 1200, height = 800, res = 120)
                                       plot(density(log2(counts(dds)[, 1] + 1)), col = "#4A90E2", lwd = 2,
                                            main = "Density Plot - Log2(Counts+1)",
                                                 xlab = "log2(Counts + 1)", ylab = "Density",
                                                      ylim = c(0, 0.3), xlim = c(0, 15))
                                                      for (i in 2:6) {
                                                        lines(density(log2(counts(dds)[, i] + 1)), col = "#4A90E2", lwd = 1.5)
                                                        }
                                                        for (i in 7:12) {
                                                          lines(density(log2(counts(dds)[, i] + 1)), col = "#E24A4A", lwd = 1.5)
                                                          }
                                                          legend("topright", legend = c("Control", "Treatment"),
                                                                 col = c("#4A90E2", "#E24A4A"), lwd = 2, cex = 0.8)
                                                                 dev.off()

                                                                 cat("📊 رسم 3: PCA Plot...\n")

                                                                 # 1.3 PCA Plot
                                                                 png("03_PCA_QC.png", width = 1200, height = 900, res = 120)
                                                                 pca_data <- plotPCA(rld, intgroup = c("condition"), returnData = TRUE)
                                                                 percentVar <- round(100 * attr(pca_data, "percentVar"))

                                                                 p_pca <- ggplot(pca_data, aes(x = PC1, y = PC2,
                                                                                               color = condition,
                                                                                                                             label = name)) +
                                                                                                                               geom_point(size = 4, alpha = 0.8) +
                                                                                                                                 geom_text(vjust = -1, size = 3, show.legend = FALSE) +
                                                                                                                                   xlab(paste0("PC1: ", percentVar[1], "% variance")) +
                                                                                                                                     ylab(paste0("PC2: ", percentVar[2], "% variance")) +
                                                                                                                                       ggtitle("PCA Plot - Sample Clustering") +
                                                                                                                                         scale_color_manual(values = c("#4A90E2", "#E24A4A")) +
                                                                                                                                           theme_bw() +
                                                                                                                                             theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))

                                                                                                                                             print(p_pca)
                                                                                                                                             dev.off()

                                                                                                                                             cat("📊 رسم 4: Sample-to-Sample Distance Heatmap...\n")

                                                                                                                                             # 1.4 Sample Distance Heatmap
                                                                                                                                             png("04_SampleDistance_QC.png", width = 1000, height = 900, res = 120)
                                                                                                                                             sampleDists <- dist(t(assay(rld)))
                                                                                                                                             sampleDistMatrix <- as.matrix(sampleDists)
                                                                                                                                             rownames(sampleDistMatrix) <- colnames(rld)
                                                                                                                                             colnames(sampleDistMatrix) <- colnames(rld)

                                                                                                                                             colors <- colorRampPalette(rev(brewer.pal(9, "Blues")))(255)
                                                                                                                                             pheatmap(sampleDistMatrix,
                                                                                                                                                      clustering_distance_rows = sampleDists,
                                                                                                                                                               clustering_distance_cols = sampleDists,
                                                                                                                                                                        col = colors,
                                                                                                                                                                                 main = "Sample-to-Sample Distance Heatmap",
                                                                                                                                                                                          fontsize = 9)
                                                                                                                                                                                          dev.off()

                                                                                                                                                                                          cat("📊 رسم 5: MA Plot...\n")

                                                                                                                                                                                          # 1.5 MA Plot
                                                                                                                                                                                          png("05_MA_Plot.png", width = 1000, height = 800, res = 120)
                                                                                                                                                                                          plotMA(res, ylim = c(-5, 5),
                                                                                                                                                                                                 main = "MA Plot - DESeq2 Results",
                                                                                                                                                                                                        colNonSig = "gray60",
                                                                                                                                                                                                               colSig = "red",
                                                                                                                                                                                                                      colLine = "blue")
                                                                                                                                                                                                                      dev.off()

                                                                                                                                                                                                                      cat("✅ اكتملت رسومات QC!\n\n")

                                                                                                                                                                                                                      # ============================================
                                                                                                                                                                                                                      # 2. DIFFERENTIAL EXPRESSION PLOTS
                                                                                                                                                                                                                      # ============================================

                                                                                                                                                                                                                      cat("📊 رسم 6: Volcano Plot...\n")

                                                                                                                                                                                                                      # 2.1 Volcano Plot
                                                                                                                                                                                                                      png("06_Volcano_DE.png", width = 1200, height = 1000, res = 120)

                                                                                                                                                                                                                      # استخدام EnhancedVolcano
                                                                                                                                                                                                                      p_volcano <- EnhancedVolcano(res,
                                                                                                                                                                                                                                      lab = rownames(res),
                                                                                                                                                                                                                                                      x = 'log2FoldChange',
                                                                                                                                                                                                                                                                      y = 'padj',
                                                                                                                                                                                                                                                                                      pCutoff = 0.05,
                                                                                                                                                                                                                                                                                                      FCcutoff = 1.0,
                                                                                                                                                                                                                                                                                                                      pointSize = 2.0,
                                                                                                                                                                                                                                                                                                                                      labSize = 3.0,
                                                                                                                                                                                                                                                                                                                                                      title = 'Volcano Plot - Differential Expression',
                                                                                                                                                                                                                                                                                                                                                                      subtitle = 'Treatment vs Control',
                                                                                                                                                                                                                                                                                                                                                                                      col = c('grey30', 'forestgreen', 'royalblue', 'red2'),
                                                                                                                                                                                                                                                                                                                                                                                                      colAlpha = 0.7,
                                                                                                                                                                                                                                                                                                                                                                                                                      legendPosition = 'right',
                                                                                                                                                                                                                                                                                                                                                                                                                                      legendLabSize = 10,
                                                                                                                                                                                                                                                                                                                                                                                                                                                      drawConnectors = TRUE,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      widthConnectors = 0.5)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      print(p_volcano)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      dev.off()

                                                                                                                                                                                                                                                                                                                                                                                                                                                                      cat("📊 رسم 7: Heatmap for Top DEGs...\n")

                                                                                                                                                                                                                                                                                                                                                                                                                                                                      # 2.2 Heatmap for Top DEGs
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      png("07_Heatmap_DEGs.png", width = 1200, height = 1000, res = 120)

                                                                                                                                                                                                                                                                                                                                                                                                                                                                      # اختيار أفضل 50 جين معنوي
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      top_genes <- head(order(res$padj), 50)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      mat <- assay(rld)[top_genes, ]

                                                                                                                                                                                                                                                                                                                                                                                                                                                                      # إضافة annotation للعينات
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      annotation_col <- data.frame(
                                                                                                                                                                                                                                                                                                                                                                                                                                                                        Condition = sample_info$condition,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                          Batch = sample_info$batch
                                                                                                                                                                                                                                                                                                                                                                                                                                                                          )
                                                                                                                                                                                                                                                                                                                                                                                                                                                                          rownames(annotation_col) <- colnames(mat)

                                                                                                                                                                                                                                                                                                                                                                                                                                                                          ann_colors <- list(
                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Condition = c(Control = "#4A90E2", Treatment = "#E24A4A"),
                                                                                                                                                                                                                                                                                                                                                                                                                                                                              Batch = c(Batch1 = "#FFD700", Batch2 = "#90EE90")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                              )

                                                                                                                                                                                                                                                                                                                                                                                                                                                                              pheatmap(mat,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       scale = "row",
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                cluster_rows = TRUE,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         cluster_cols = TRUE,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  annotation_col = annotation_col,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           annotation_colors = ann_colors,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    color = colorRampPalette(c("navy", "white", "firebrick3"))(100),
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             main = "Heatmap - Top 50 Differentially Expressed Genes",
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      fontsize = 9,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               show_rownames = FALSE)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               dev.off()

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               cat("📊 رسم 8: Bar Plot (Up vs Down)...\n")

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               # 2.3 Bar Plot
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               png("08_Bar_DE.png", width = 1000, height = 800, res = 120)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               up_count <- sum(res$padj < 0.05 & res$log2FoldChange > 1, na.rm = TRUE)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               down_count <- sum(res$padj < 0.05 & res$log2FoldChange < -1, na.rm = TRUE)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ns_count <- nrow(res) - up_count - down_count

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               bar_data <- data.frame(
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 Category = c("Upregulated", "Downregulated", "Not Significant"),
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   Count = c(up_count, down_count, ns_count),
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     Color = c("#E24A4A", "#4A90E2", "grey70")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     )

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     p_bar <- ggplot(bar_data, aes(x = Category, y = Count, fill = Category)) +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       geom_bar(stat = "identity", alpha = 0.8) +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         geom_text(aes(label = Count), vjust = -0.5, size = 5) +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           scale_fill_manual(values = c("#E24A4A", "#4A90E2", "grey70")) +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             ggtitle("Differential Expression - Gene Counts") +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               xlab("") + ylab("Number of Genes") +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 theme_bw() +
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   theme(legend.position = "none",
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           plot.title = element_text(hjust = 0.5, face = "bold", size = 14))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           print(p_bar)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           dev.off()

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           cat("✅ اكتملت رسومات Differential Expression!\n\n")

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           # ============================================
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           # عرض قائمة الملفات المحفوظة
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           # ============================================
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           cat("📁 الملفات المحفوظة:\n")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           cat("========================================\n")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           files <- list.files(pattern = "\\.png$")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           for (f in files) {
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             cat("  ✓", f, "\n")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             }
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             cat("========================================\n")